# rank0-only-side-effects — faded example 2: Rank 0 downloads, then barrier, then all ranks read

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `rank0-only-side-effects`. Running the beacon reports progress on the `Distributed: rank-0-only side effects` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: rank-0-only side effects` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`rank0-only-side-effects`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "rank0-only-side-effects"
DD_SUBTOPIC = "Distributed: rank-0-only side effects"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

A common startup pattern is for rank 0 to download or prepare shared data, followed by a barrier that holds all other ranks until rank 0 finishes, after which every rank can safely read the data. The ordering guarantee is critical: without the barrier, rank 1 might try to read before rank 0 has finished writing.

## Faded exercise 2

### Exercise — Rank 0 downloads, barrier, all ranks read

Complete `init_data(rank, world_size, dist_mod, downloader, reader, log)`. Rank 0 calls `downloader()`, then every rank calls `dist_mod.barrier()`, then every rank calls `reader()`.

Fill in the rank-0 download guard.

**Fill in:** Write the rank-0 guard that calls downloader() (and logs) before the barrier.

In [ ]:
def init_data(rank, world_size, dist_mod, downloader, reader, log):
    if rank == 0:
        downloader()
        log('rank0-downloaded')
    dist_mod.barrier()
    value = reader()
    log(f'rank{rank}-read-{value}')
    return value

# Test
class MockDist:
    def barrier(self): pass

download_count = [0]
read_count = [0]
log_entries = []

def dl(): download_count[0] += 1
def rd(): read_count[0] += 1; return 'data'
def lg(s): log_entries.append(s)

dist = MockDist()
for r in range(3):
    init_data(r, 3, dist, dl, rd, lg)
print(download_count[0], read_count[0])


def _test():
    class MockDist:
        def __init__(self): self.barrier_calls = 0
        def barrier(self): self.barrier_calls += 1
    download_count = [0]
    log_entries = []
    def dl(): download_count[0] += 1
    def rd(): return 'dataset'
    def lg(s): log_entries.append(s)
    dist = MockDist()
    for r in range(4):
        val = init_data(r, 4, dist, dl, rd, lg)
        assert val == 'dataset'
    assert download_count[0] == 1, f'expected 1 download, got {download_count[0]}'
    assert dist.barrier_calls == 4, f'barrier should be called 4 times (once per rank)'
    download_log_idx = log_entries.index('rank0-downloaded')
    for entry in log_entries:
        if 'read' in entry:
            assert log_entries.index(entry) > download_log_idx, 'download must precede all reads'


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
def init_data(rank, world_size, dist_mod, downloader, reader, log):
    if rank == 0:
        downloader()
        log('rank0-downloaded')
    dist_mod.barrier()
    value = reader()
    log(f'rank{rank}-read-{value}')
    return value

# Test
class MockDist:
    def barrier(self): pass

download_count = [0]
read_count = [0]
log_entries = []

def dl(): download_count[0] += 1
def rd(): read_count[0] += 1; return 'data'
def lg(s): log_entries.append(s)

dist = MockDist()
for r in range(3):
    init_data(r, 3, dist, dl, rd, lg)
print(download_count[0], read_count[0])
```
</details>